# Multimodal Smart Contract Vulnerability Detection

This notebook builds on `exploratory_data_analysis.ipynb` and implements:

1. **Data loading and binary label construction** – load the dataset and map the multi-label vulnerability annotations to a single binary flag (safe / vulnerable).
2. **Modality preprocessing and BERT embeddings** – clean source-code text and extract opcode-frequency features from bytecode; encode source code with CodeBERT.
3. **Fusion network + classifier training** – combine the two modalities in a late-fusion neural network and train a binary classifier.

## 0 · Install Dependencies

In [ ]:
!pip install -q datasets transformers torch scikit-learn hexbytes pyevmasm

## 1 · Data Loading and Binary Label Construction

The dataset has six vulnerability labels: `access-control`, `arithmetic`, `other`, `reentrancy`, `safe`, `unchecked-calls`.
A contract is **safe** when its only label is `4` (safe); every other contract is **vulnerable**.
We collapse the multi-label vector into a single binary target:

| binary label | meaning |
|---|---|
| `0` | safe – no vulnerability detected |
| `1` | vulnerable – at least one vulnerability present |

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from hexbytes import HexBytes
from datasets import load_dataset

SAFE_LABEL = 4  # index that corresponds to the 'safe' class

LABEL_NAMES = {
    0: 'access-control',
    1: 'arithmetic',
    2: 'other',
    3: 'reentrancy',
    4: 'safe',
    5: 'unchecked-calls',
}


def make_binary_label(slither_labels: list) -> int:
    """Return 0 if the only label is 'safe' (index 4), else 1 (vulnerable)."""
    return 0 if slither_labels == [SAFE_LABEL] else 1


# ------------------------------------------------------------------
# Load all three splits from HuggingFace Hub
# ------------------------------------------------------------------
print("Loading dataset splits …")
train_hf = load_dataset(
    "mwritescode/slither-audited-smart-contracts",
    "big-multilabel",
    split="train",
    trust_remote_code=True,
)
val_hf = load_dataset(
    "mwritescode/slither-audited-smart-contracts",
    "big-multilabel",
    split="validation",
    trust_remote_code=True,
)
test_hf = load_dataset(
    "mwritescode/slither-audited-smart-contracts",
    "big-multilabel",
    split="test",
    trust_remote_code=True,
)
print(f"Train: {len(train_hf):,}  Val: {len(val_hf):,}  Test: {len(test_hf):,}")

In [ ]:
def hf_split_to_df(hf_split, split_name: str) -> pd.DataFrame:
    """Convert a HuggingFace Dataset split to a pandas DataFrame with a binary label."""
    records = []
    for row in hf_split:
        records.append(
            {
                "address": row["address"],
                "source_code": row["source_code"],
                "bytecode": row["bytecode"],
                "slither": row["slither"],
                "label": make_binary_label(row["slither"]),
                "split": split_name,
            }
        )
    return pd.DataFrame(records)


train_df = hf_split_to_df(train_hf, "train")
val_df = hf_split_to_df(val_hf, "val")
test_df = hf_split_to_df(test_hf, "test")

full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
print(full_df[["split", "label"]].value_counts().sort_index())

In [ ]:
# ------------------------------------------------------------------
# Visualise binary-label distribution per split
# ------------------------------------------------------------------
_, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
label_map = {0: "safe", 1: "vulnerable"}

for ax, (name, grp) in zip(axes, full_df.groupby("split")):
    counts = grp["label"].map(label_map).value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=ax)
    ax.set_title(f"{name} split")
    ax.set_xlabel("Binary label")
    ax.set_ylabel("Count")
    for bar, val in zip(ax.patches, counts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 5,
            str(val),
            ha="center",
        )

plt.suptitle("Binary label distribution per split", fontsize=14)
plt.tight_layout()
plt.show()

## 2 · Modality Preprocessing and BERT Embeddings

We work with two modalities:

| Modality | Raw input | Preprocessing | Encoder |
|---|---|---|---|
| Source code | Solidity text | Strip comments, truncate | CodeBERT (`microsoft/codebert-base`) → 768-d CLS vector |
| Bytecode | Hex string | Disassemble to opcodes → normalised frequency vector | Fixed-size opcode histogram |

### 2.1 Source-code preprocessing

In [ ]:
def remove_comments(code: str) -> str:
    """Remove single-line (//) and multi-line (/* */) comments from Solidity source."""
    pattern = r'(".*?"|\'.*?\')|(/\*.*?\*/|//[^\r\n]*$)'
    regex = re.compile(pattern, re.MULTILINE | re.DOTALL)

    def _replacer(match):
        return "" if match.group(2) is not None else match.group(1)

    return regex.sub(_replacer, code)


def clean_source(code: str) -> str:
    """Strip comments and normalise whitespace."""
    code = remove_comments(code)
    code = re.sub(r"\s+", " ", code).strip()
    return code


# Quick sanity check
sample = train_df["source_code"].iloc[0]
cleaned = clean_source(sample)
print(f"Original length : {len(sample):,} chars")
print(f"Cleaned length  : {len(cleaned):,} chars")
print("Snippet:", cleaned[:300])

### 2.2 Bytecode preprocessing – opcode frequency histogram

In [ ]:
try:
    import pyevmasm as evmasm
    EVMASM_AVAILABLE = True
except ImportError:
    EVMASM_AVAILABLE = False
    print("pyevmasm not available – falling back to raw byte-bigram features.")

# EVM opcode names (256 slots, 0x00–0xff)
NUM_OPCODES = 256


def bytecode_to_opcode_freq(hex_bytecode: str) -> np.ndarray:
    """
    Convert a hex bytecode string to a normalised opcode frequency vector
    of length NUM_OPCODES (one entry per possible opcode value 0x00–0xff).

    If the bytecode is empty / unavailable, returns a zero vector.
    """
    freq = np.zeros(NUM_OPCODES, dtype=np.float32)

    if not hex_bytecode or hex_bytecode in ("0x", "0x0", ""):
        return freq

    try:
        raw_bytes = bytes(HexBytes(hex_bytecode))
    except Exception:
        return freq

    if EVMASM_AVAILABLE:
        try:
            for instr in evmasm.disassemble_all(raw_bytes):
                freq[instr.opcode] += 1
        except Exception:
            # Fallback: count raw bytes
            for b in raw_bytes:
                freq[b] += 1
    else:
        for b in raw_bytes:
            freq[b] += 1

    total = freq.sum()
    if total > 0:
        freq /= total  # L1-normalise so length differences don't dominate

    return freq


# Sanity check
sample_bc = train_df["bytecode"].iloc[0]
opcode_vec = bytecode_to_opcode_freq(sample_bc)
print(f"Opcode frequency vector shape: {opcode_vec.shape}")
print(f"Non-zero entries: {np.count_nonzero(opcode_vec)}")
print(f"Sum (should be ~1): {opcode_vec.sum():.4f}")

### 2.3 BERT embeddings for source code

We use `microsoft/codebert-base` – a RoBERTa model pre-trained on code – to generate a 768-dimensional CLS-token embedding for each contract's (cleaned) source code.

> **Note**: Generating embeddings for the full dataset is compute-intensive. The cells below demonstrate the pipeline on a small `DEMO_SIZE` subset. Remove the subsetting to run on the full data (requires GPU).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "microsoft/codebert-base"
MAX_SEQ_LEN = 512  # CodeBERT maximum
DEMO_SIZE = 200    # set to None to process the full dataset
BATCH_SIZE = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading tokeniser and model ({MODEL_NAME}) …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
bert_model.eval()
print("Done.")

In [ ]:
def get_bert_embeddings(texts: list, batch_size: int = BATCH_SIZE) -> np.ndarray:
    """
    Encode a list of strings with CodeBERT and return the CLS-token embeddings
    as an (N, 768) float32 numpy array.
    """
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            outputs = bert_model(**encoded)

        # CLS token embedding: shape (batch, 768)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)

        if (i // batch_size) % 5 == 0:
            print(f"  Encoded {min(i + batch_size, len(texts))}/{len(texts)} samples …")

    return np.vstack(all_embeddings)

In [ ]:
# ------------------------------------------------------------------
# Build feature matrices for each split
# ------------------------------------------------------------------

def prepare_split(df: pd.DataFrame, demo_size=None):
    """Compute source-code BERT embeddings + bytecode opcode features for a split."""
    if demo_size is not None:
        df = df.head(demo_size).copy()

    print(f"  Preprocessing {len(df)} samples …")

    # Source-code cleaning
    cleaned_codes = df["source_code"].apply(clean_source).tolist()

    # BERT embeddings (source code)
    print("  Generating BERT embeddings …")
    X_text = get_bert_embeddings(cleaned_codes)  # (N, 768)

    # Opcode frequency features (bytecode)
    print("  Computing opcode frequency vectors …")
    X_bytecode = np.vstack(
        df["bytecode"].apply(bytecode_to_opcode_freq).tolist()
    )  # (N, 256)

    y = df["label"].values.astype(np.int64)  # (N,)

    return X_text, X_bytecode, y


print("Preparing TRAIN split …")
X_train_text, X_train_bc, y_train = prepare_split(train_df, DEMO_SIZE)

print("\nPreparing VAL split …")
X_val_text, X_val_bc, y_val = prepare_split(val_df, DEMO_SIZE)

print("\nPreparing TEST split …")
X_test_text, X_test_bc, y_test = prepare_split(test_df, DEMO_SIZE)

print(f"\nText feature shapes  — train: {X_train_text.shape}, val: {X_val_text.shape}, test: {X_test_text.shape}")
print(f"Bytecode feature shapes — train: {X_train_bc.shape}, val: {X_val_bc.shape}, test: {X_test_bc.shape}")

## 3 · Fusion Network and Classifier Training

We implement a **late-fusion** architecture:

```
Source Code (768-d)  ──► TextBranch (MLP) ──┐
                                             ├──► concat ──► FusionHead ──► sigmoid ──► binary label
Bytecode (256-d)     ──► BytecodeBranch     ──┘
```

Each branch independently maps its input to a lower-dimensional space; the two representations are concatenated and passed to a final classification head.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report


# ------------------------------------------------------------------
# Dataset
# ------------------------------------------------------------------

class MultimodalContractDataset(Dataset):
    """Pairs (text_embedding, bytecode_features, binary_label)."""

    def __init__(self, X_text: np.ndarray, X_bc: np.ndarray, y: np.ndarray):
        self.X_text = torch.tensor(X_text, dtype=torch.float32)
        self.X_bc = torch.tensor(X_bc, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_text[idx], self.X_bc[idx], self.y[idx]


train_dataset = MultimodalContractDataset(X_train_text, X_train_bc, y_train)
val_dataset   = MultimodalContractDataset(X_val_text,   X_val_bc,   y_val)
test_dataset  = MultimodalContractDataset(X_test_text,  X_test_bc,  y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32)
test_loader  = DataLoader(test_dataset,  batch_size=32)

print(f"Dataset sizes — train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")

In [ ]:
# ------------------------------------------------------------------
# Fusion network architecture
# ------------------------------------------------------------------

class FusionNet(nn.Module):
    """
    Late-fusion binary classifier for smart-contract vulnerability detection.

    Parameters
    ----------
    text_dim      : dimensionality of the BERT CLS embeddings (default 768)
    bytecode_dim  : number of opcode frequency bins (default 256)
    text_hidden   : hidden size of the text branch MLP
    bc_hidden     : hidden size of the bytecode branch MLP
    fusion_hidden : hidden size of the fusion head
    dropout       : dropout probability applied in the fusion head
    """

    def __init__(
        self,
        text_dim: int = 768,
        bytecode_dim: int = 256,
        text_hidden: int = 256,
        bc_hidden: int = 128,
        fusion_hidden: int = 128,
        dropout: float = 0.3,
    ):
        super().__init__()

        # Source-code branch
        self.text_branch = nn.Sequential(
            nn.Linear(text_dim, text_hidden),
            nn.ReLU(),
            nn.LayerNorm(text_hidden),
        )

        # Bytecode branch
        self.bc_branch = nn.Sequential(
            nn.Linear(bytecode_dim, bc_hidden),
            nn.ReLU(),
            nn.LayerNorm(bc_hidden),
        )

        # Fusion head
        self.fusion_head = nn.Sequential(
            nn.Linear(text_hidden + bc_hidden, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, 2),
        )

    def forward(self, x_text: torch.Tensor, x_bc: torch.Tensor) -> torch.Tensor:
        text_feat = self.text_branch(x_text)     # (B, text_hidden)
        bc_feat   = self.bc_branch(x_bc)         # (B, bc_hidden)
        fused     = torch.cat([text_feat, bc_feat], dim=1)  # (B, text_hidden+bc_hidden)
        return self.fusion_head(fused)            # (B, 2)  – raw logits


model = FusionNet().to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ------------------------------------------------------------------
# Training setup
# ------------------------------------------------------------------

# Handle class imbalance with per-class weights for CrossEntropyLoss.
# The minority (vulnerable) class gets a higher weight proportional to
# how much more frequent the majority (safe) class is.
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
class_weights = torch.tensor(
    [1.0, n_neg / max(n_pos, 1)], dtype=torch.float32
).to(device)
print(f"Training label distribution — safe (0): {n_neg}, vulnerable (1): {n_pos}")
print(f"Class weights: safe={class_weights[0]:.3f}, vulnerable={class_weights[1]:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)


In [ ]:
# ------------------------------------------------------------------
# Helper: evaluate on a DataLoader
# ------------------------------------------------------------------

def evaluate(model, loader):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():
        for x_text, x_bc, y in loader:
            x_text, x_bc = x_text.to(device), x_bc.to(device)
            logits = model(x_text, x_bc)          # (B, 2)
            probs  = torch.softmax(logits, dim=1)[:, 1]  # P(vulnerable)
            preds  = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
            all_labels.extend(y.numpy().tolist())

    acc    = accuracy_score(all_labels, all_preds)
    f1     = f1_score(all_labels, all_preds, zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float("nan")

    return acc, f1, auc

In [ ]:
# ------------------------------------------------------------------
# Training loop
# ------------------------------------------------------------------

NUM_EPOCHS = 15

history = {"train_loss": [], "val_acc": [], "val_f1": [], "val_auc": []}
best_val_f1 = -1.0
best_model_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for x_text, x_bc, y in train_loader:
        x_text, x_bc, y = x_text.to(device), x_bc.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x_text, x_bc)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(y)

    scheduler.step()

    avg_loss = running_loss / len(train_dataset)
    val_acc, val_f1, val_auc = evaluate(model, val_loader)

    history["train_loss"].append(avg_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)
    history["val_auc"].append(val_auc)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"loss: {avg_loss:.4f} | "
        f"val acc: {val_acc:.4f} | "
        f"val F1: {val_f1:.4f} | "
        f"val AUC: {val_auc:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

print(f"\nBest validation F1: {best_val_f1:.4f}")

In [ ]:
# ------------------------------------------------------------------
# Plot training curves
# ------------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

epochs_range = range(1, NUM_EPOCHS + 1)

axes[0].plot(epochs_range, history["train_loss"], marker="o")
axes[0].set_title("Training loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")

axes[1].plot(epochs_range, history["val_acc"], marker="o", label="Accuracy")
axes[1].plot(epochs_range, history["val_f1"],  marker="s", label="F1")
axes[1].set_title("Validation accuracy and F1")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(epochs_range, history["val_auc"], marker="^", color="green")
axes[2].set_title("Validation ROC-AUC")
axes[2].set_xlabel("Epoch")

plt.suptitle("FusionNet training history", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Final evaluation on the test split using best checkpoint
# ------------------------------------------------------------------

if best_model_state is not None:
    model.load_state_dict(best_model_state)

test_acc, test_f1, test_auc = evaluate(model, test_loader)
print("=== Test-set results (best checkpoint) ===")
print(f"  Accuracy : {test_acc:.4f}")
print(f"  F1 score : {test_f1:.4f}")
print(f"  ROC-AUC  : {test_auc:.4f}")

# Detailed classification report
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x_text, x_bc, y in test_loader:
        logits = model(x_text.to(device), x_bc.to(device))
        all_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        all_labels.extend(y.numpy().tolist())

print()
print(classification_report(
    all_labels, all_preds,
    target_names=["safe", "vulnerable"],
    zero_division=0,
))

In [ ]:
# ------------------------------------------------------------------
# Optional: save the best model weights
# ------------------------------------------------------------------

# torch.save(best_model_state, "fusion_net_best.pt")
# print("Model weights saved to fusion_net_best.pt")